# Experiment 11 — LightProtoSSM + MLP Probe Pipeline

**Source:** Adaptation of `discussion/0-932-test-mod-version-4.ipynb` (public LB 0.932).

**Pipeline:**
- Perch v2 ONNX embeddings + logits (frozen)
- Site/hour prior adjustment
- LightProtoSSM (SSM with prototype distance)
- sklearn MLP probes on PCA-compressed embeddings
- Ensemble + adaptive smoothing

**Evaluation:** GroupKFold (n=3) OOF on 59 fully-labeled soundscape files.

In [1]:
import os, re, gc, time, warnings, concurrent.futures
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import soundfile as sf
import torch
import torch.nn as nn
import torch.nn.functional as F
import onnxruntime as ort
from pathlib import Path
from tqdm.auto import tqdm
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import GroupKFold
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier

torch.manual_seed(42)
np.random.seed(42)

PROJECT_ROOT = Path('/Users/jjannik/Development/kaggle/birdclef')
BASE      = PROJECT_ROOT / 'data'
MODEL_DIR = BASE / 'models' / 'perch_tf'
ONNX_PATH = BASE / 'models' / 'perch' / 'perch_v2_no_dft.onnx'
CACHE_DIR = BASE / 'cache'
CACHE_DIR.mkdir(parents=True, exist_ok=True)

CACHE_META = CACHE_DIR / 'exp11_perch_meta.csv'
CACHE_NPZ  = CACHE_DIR / 'exp11_perch_arrays.npz'

SR             = 32_000
WINDOW_SEC     = 5
WINDOW_SAMPLES = SR * WINDOW_SEC   # 160000
FILE_SAMPLES   = 60 * SR
N_WINDOWS      = 12

N_OOF_SPLITS   = 3

_WALL_START = time.time()
print('Setup done')

Setup done


In [2]:
# Data loading
taxonomy          = pd.read_csv(BASE / 'taxonomy.csv')
sample_sub        = pd.read_csv(BASE / 'sample_submission.csv')
soundscape_labels = pd.read_csv(BASE / 'train_soundscapes_labels.csv')

PRIMARY_LABELS = sample_sub.columns[1:].tolist()
N_CLASSES      = len(PRIMARY_LABELS)
label_to_idx   = {c: i for i, c in enumerate(PRIMARY_LABELS)}

FNAME_RE = re.compile(r'BC2026_(?:Train|Test)_(\d+)_(S\d+)_(\d{8})_(\d{6})\.ogg')

def parse_fname(name):
    m = FNAME_RE.match(name)
    if not m:
        return {'site': 'unknown', 'hour_utc': -1}
    _, site, _, hms = m.groups()
    return {'site': site, 'hour_utc': int(hms[:2])}

def union_labels(series):
    out = set()
    for x in series:
        if pd.notna(x):
            for t in str(x).split(';'):
                t = t.strip()
                if t:
                    out.add(t)
    return sorted(out)

sc = (
    soundscape_labels
    .groupby(['filename', 'start', 'end'])['primary_label']
    .apply(union_labels)
    .reset_index(name='label_list')
)
sc['end_sec'] = pd.to_timedelta(sc['end']).dt.total_seconds().astype(int)
sc['row_id'] = (
    sc['filename'].str.replace('.ogg', '', regex=False)
    + '_' + sc['end_sec'].astype(str)
)
_meta = sc['filename'].apply(parse_fname).apply(pd.Series)
sc = pd.concat([sc, _meta], axis=1)

Y_SC = np.zeros((len(sc), N_CLASSES), dtype=np.uint8)
for i, lbls in enumerate(sc['label_list']):
    for lbl in lbls:
        if lbl in label_to_idx:
            Y_SC[i, label_to_idx[lbl]] = 1

windows_per_file = sc.groupby('filename').size()
LABEL_WINDOWS = int(windows_per_file.mode().iloc[0])
full_files = sorted(windows_per_file[windows_per_file == LABEL_WINDOWS].index.tolist())

sc['fully_labeled'] = sc['filename'].isin(full_files)
full_rows = (
    sc[sc['fully_labeled']]
    .sort_values(['filename', 'end_sec'])
    .reset_index(drop=False)
)
Y_FULL = Y_SC[full_rows['index'].to_numpy()]

print(f'Classes: {N_CLASSES} | Fully-labeled files: {len(full_files)}')
print(f'Full-file windows: {len(full_rows)} | Active classes: {int((Y_FULL.sum(0) > 0).sum())}')

Classes: 234 | Fully-labeled files: 59
Full-file windows: 708 | Active classes: 71


In [3]:
# Load Perch ONNX model
_so = ort.SessionOptions()
_so.intra_op_num_threads = 4
ONNX_SESSION = ort.InferenceSession(
    str(ONNX_PATH),
    sess_options=_so,
    providers=['CPUExecutionProvider'],
)
ONNX_INPUT_NAME = ONNX_SESSION.get_inputs()[0].name
ONNX_OUT_MAP    = {o.name: i for i, o in enumerate(ONNX_SESSION.get_outputs())}
print('ONNX inputs :', [(i.name, i.shape) for i in ONNX_SESSION.get_inputs()])
print('ONNX outputs:', list(ONNX_OUT_MAP.keys()))

# Perch label mapping
bc_labels = (
    pd.read_csv(MODEL_DIR / 'assets' / 'labels.csv')
    .reset_index()
    .rename(columns={'index': 'bc_index', 'inat2024_fsd50k': 'scientific_name'})
)
NO_LABEL = len(bc_labels)

mapping = (
    taxonomy
    .merge(bc_labels.rename(columns={'scientific_name': 'scientific_name'}),
           on='scientific_name', how='left')
)
mapping['bc_index'] = mapping['bc_index'].fillna(NO_LABEL).astype(int)
lbl2bc = mapping.set_index('primary_label')['bc_index']

BC_INDICES    = np.array([int(lbl2bc.loc[c]) for c in PRIMARY_LABELS], dtype=np.int32)
MAPPED_MASK   = BC_INDICES != NO_LABEL
MAPPED_POS    = np.where(MAPPED_MASK)[0].astype(np.int32)
MAPPED_BC_IDX = BC_INDICES[MAPPED_MASK].astype(np.int32)

print(f'Mapped: {MAPPED_MASK.sum()} / {N_CLASSES} species have a Perch logit')

ONNX inputs : [('inputs', ['batch', 160000])]
ONNX outputs: ['embedding', 'spatial_embedding', 'spectrogram', 'label']


Mapped: 203 / 234 species have a Perch logit


In [4]:
# Genus proxy for unmapped species
import re as _re

UNMAPPED_POS  = np.where(~MAPPED_MASK)[0].astype(np.int32)
CLASS_NAME_MAP = taxonomy.set_index('primary_label')['class_name'].to_dict()

proxy_map = {}
unmapped_df = taxonomy[taxonomy['primary_label'].isin([PRIMARY_LABELS[i] for i in UNMAPPED_POS])].copy()

for _, row in unmapped_df.iterrows():
    target = row['primary_label']
    genus  = str(row['scientific_name']).split()[0]
    hits = bc_labels[
        bc_labels['scientific_name'].astype(str).str.match(rf'^{_re.escape(genus)}\s', na=False)
    ]
    if len(hits) > 0:
        proxy_map[label_to_idx[target]] = hits['bc_index'].astype(int).tolist()

PROXY_TAXA = {'Amphibia', 'Insecta', 'Aves'}
proxy_map  = {
    idx: bc_idxs for idx, bc_idxs in proxy_map.items()
    if CLASS_NAME_MAP.get(PRIMARY_LABELS[idx]) in PROXY_TAXA
}
print(f'Unmapped: {len(UNMAPPED_POS)} | With genus proxy: {len(proxy_map)}')

Unmapped: 31 | With genus proxy: 3


In [5]:
# Perch inference engine
def read_60s(path):
    y, sr = sf.read(path, dtype='float32', always_2d=False)
    if y.ndim == 2: y = y.mean(axis=1)
    if len(y) < FILE_SAMPLES: y = np.pad(y, (0, FILE_SAMPLES - len(y)))
    else:                      y = y[:FILE_SAMPLES]
    return y

def run_perch(paths, batch_files=8, verbose=True):
    paths  = [Path(p) for p in paths]
    n_rows = len(paths) * N_WINDOWS

    row_ids   = np.empty(n_rows, dtype=object)
    filenames = np.empty(n_rows, dtype=object)
    sites     = np.empty(n_rows, dtype=object)
    hours     = np.zeros(n_rows, dtype=np.int16)
    scores    = np.zeros((n_rows, N_CLASSES), dtype=np.float32)
    embs      = np.zeros((n_rows, 1536), dtype=np.float32)

    wr  = 0
    itr = tqdm(range(0, len(paths), batch_files), desc='Perch') if verbose else range(0, len(paths), batch_files)

    with concurrent.futures.ThreadPoolExecutor(max_workers=4) as io_executor:
        next_paths   = paths[0:batch_files]
        future_audio = [io_executor.submit(read_60s, p) for p in next_paths]

        for start in itr:
            batch_paths  = next_paths
            batch_n      = len(batch_paths)
            batch_audio  = [f.result() for f in future_audio]

            next_start = start + batch_files
            if next_start < len(paths):
                next_paths   = paths[next_start:next_start + batch_files]
                future_audio = [io_executor.submit(read_60s, p) for p in next_paths]

            x  = np.empty((batch_n * N_WINDOWS, WINDOW_SAMPLES), dtype=np.float32)
            br = wr

            for bi, path in enumerate(batch_paths):
                y    = batch_audio[bi]
                meta = parse_fname(path.name)
                stem = path.stem
                x[bi * N_WINDOWS:(bi + 1) * N_WINDOWS] = y.reshape(N_WINDOWS, WINDOW_SAMPLES)
                row_ids  [wr:wr + N_WINDOWS] = [f'{stem}_{t}' for t in range(5, 65, 5)]
                filenames[wr:wr + N_WINDOWS] = path.name
                sites    [wr:wr + N_WINDOWS] = meta['site']
                hours    [wr:wr + N_WINDOWS] = meta['hour_utc']
                wr += N_WINDOWS

            outs   = ONNX_SESSION.run(None, {ONNX_INPUT_NAME: x})
            logits = outs[ONNX_OUT_MAP['label']].astype(np.float32)
            emb    = outs[ONNX_OUT_MAP['embedding']].astype(np.float32)

            scores[br:wr, MAPPED_POS] = logits[:, MAPPED_BC_IDX]
            embs  [br:wr]             = emb

            for pos_idx, bc_idxs in proxy_map.items():
                bc_arr = np.array(bc_idxs, dtype=np.int32)
                scores[br:wr, pos_idx] = logits[:, bc_arr].max(axis=1)

            del x, logits, emb, batch_audio
            gc.collect()

    meta_df = pd.DataFrame({'row_id': row_ids, 'filename': filenames,
                             'site': sites, 'hour_utc': hours})
    return meta_df, scores, embs

print('Perch engine ready')

Perch engine ready


In [6]:
# Build or load Perch training cache
SCORE_KEYS = ['scores', 'sc', 'logits', 'arr_0']
EMB_KEYS   = ['embs', 'emb', 'embeddings', 'arr_1']

def _pick_array(arr, candidates, ncols):
    for k in candidates:
        if k in arr.files:
            return arr[k], k
    for k in arr.files:
        v = arr[k]
        if v.ndim == 2 and v.shape[1] == ncols:
            return v, k
    raise KeyError(f'Keys {candidates} not found. Available: {arr.files}')

if CACHE_META.exists() and CACHE_NPZ.exists():
    print('Loading cached Perch features...')
else:
    print(f'Building Perch cache from {len(full_files)} files...')
    train_paths = [BASE / 'train_soundscapes' / fn for fn in full_files]
    t0 = time.time()
    meta_built, sc_built, emb_built = run_perch(train_paths, batch_files=8, verbose=True)
    print(f'  Done in {time.time()-t0:.1f}s')
    meta_built.to_csv(CACHE_META, index=False)
    np.savez(CACHE_NPZ,
             scores=sc_built.astype(np.float32),
             embs=emb_built.astype(np.float32),
             primary_labels=np.array(PRIMARY_LABELS))
    print(f'  Cache saved to {CACHE_DIR}')

meta_tr = pd.read_csv(CACHE_META)
_arr    = np.load(CACHE_NPZ)
sc_tr_raw,  _ = _pick_array(_arr, SCORE_KEYS, N_CLASSES)
emb_tr_raw, _ = _pick_array(_arr, EMB_KEYS,   1536)
sc_tr  = sc_tr_raw.astype(np.float32)
emb_tr = emb_tr_raw.astype(np.float32)

row_id_to_index = full_rows.set_index('row_id')['index']
Y_FULL_aligned = Y_SC[row_id_to_index.loc[meta_tr['row_id']].to_numpy()]

print(f'sc_tr: {sc_tr.shape}  emb_tr: {emb_tr.shape}  Y_FULL_aligned: {Y_FULL_aligned.shape}')

Building Perch cache from 59 files...


Perch:   0%|          | 0/8 [00:00<?, ?it/s]

  Done in 16.0s
  Cache saved to /Users/jjannik/Development/kaggle/birdclef/data/cache
sc_tr: (708, 234)  emb_tr: (708, 1536)  Y_FULL_aligned: (708, 234)


In [7]:
# Metrics
def macro_auc(y_true, y_score):
    keep = y_true.sum(axis=0) > 0
    return roc_auc_score(y_true[:, keep], y_score[:, keep], average='macro')

def padded_cmap(y_true, y_pred, pad=5):
    aps = []
    for c in range(y_true.shape[1]):
        yt = np.concatenate([y_true[:, c], np.ones(pad)])
        yp = np.concatenate([y_pred[:, c], np.ones(pad)])
        aps.append(average_precision_score(yt, yp))
    return float(np.mean(aps))

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -30, 30)))

print('Metrics ready')

Metrics ready


In [8]:
# Helper functions

def build_prior_tables(sc_df, Y_labels):
    sc_df = sc_df.reset_index(drop=True)
    global_p = Y_labels.mean(axis=0).astype(np.float32)
    site_keys = sorted(sc_df['site'].dropna().astype(str).unique())
    site_to_i = {k: i for i, k in enumerate(site_keys)}
    site_p    = np.zeros((len(site_keys), Y_labels.shape[1]), dtype=np.float32)
    site_n    = np.zeros(len(site_keys), dtype=np.float32)
    for s in site_keys:
        i = site_to_i[s]; mask = sc_df['site'].astype(str).values == s
        site_n[i] = mask.sum(); site_p[i] = Y_labels[mask].mean(axis=0)
    hour_keys = sorted(sc_df['hour_utc'].dropna().astype(int).unique())
    hour_to_i = {h: i for i, h in enumerate(hour_keys)}
    hour_p    = np.zeros((len(hour_keys), Y_labels.shape[1]), dtype=np.float32)
    hour_n    = np.zeros(len(hour_keys), dtype=np.float32)
    for h in hour_keys:
        i = hour_to_i[h]; mask = sc_df['hour_utc'].astype(int).values == h
        hour_n[i] = mask.sum(); hour_p[i] = Y_labels[mask].mean(axis=0)
    return {'global_p': global_p,
            'site_to_i': site_to_i, 'site_p': site_p, 'site_n': site_n,
            'hour_to_i': hour_to_i, 'hour_p': hour_p, 'hour_n': hour_n}

def apply_prior(scores, sites, hours, tables, lambda_prior=0.10):
    eps = 1e-4; n = len(scores); out = scores.copy()
    p = np.tile(tables['global_p'], (n, 1))
    for i, h in enumerate(hours):
        h = int(h)
        if h in tables['hour_to_i']:
            j = tables['hour_to_i'][h]; nh = tables['hour_n'][j]
            w = nh / (nh + 8.0)
            p[i] = w * tables['hour_p'][j] + (1 - w) * tables['global_p']
    for i, s in enumerate(sites):
        s = str(s)
        if s in tables['site_to_i']:
            j = tables['site_to_i'][s]; ns = tables['site_n'][j]
            w = ns / (ns + 8.0)
            p[i] = w * tables['site_p'][j] + (1 - w) * p[i]
    p = np.clip(p, eps, 1 - eps)
    out += lambda_prior * (np.log(p) - np.log1p(-p))
    return out.astype(np.float32)

def adaptive_delta_smooth(probs, n_windows=N_WINDOWS, base_alpha=0.15):
    N, C = probs.shape
    result = probs.copy()
    view   = probs.reshape(-1, n_windows, C)
    out    = result.reshape(-1, n_windows, C)
    for t in range(n_windows):
        conf  = view[:, t, :].max(axis=-1, keepdims=True)
        alpha = base_alpha * (1.0 - conf)
        if t == 0:
            neighbor_avg = (view[:, t, :] + view[:, t+1, :]) / 2.0
        elif t == n_windows - 1:
            neighbor_avg = (view[:, t-1, :] + view[:, t, :]) / 2.0
        else:
            neighbor_avg = (view[:, t-1, :] + view[:, t+1, :]) / 2.0
        out[:, t, :] = (1.0 - alpha) * view[:, t, :] + alpha * neighbor_avg
    return result

def file_confidence_scale(probs, n_windows=N_WINDOWS, top_k=2, power=0.05):
    N, C = probs.shape
    view = probs.reshape(-1, n_windows, C)
    sorted_v = np.sort(view, axis=1)
    top_k_mean = sorted_v[:, -top_k:, :].mean(axis=1, keepdims=True)
    scale  = np.power(top_k_mean, power)
    return (view * scale).reshape(N, C)

# Per-taxon temperature
TEXTURE_TAXA = {'Amphibia', 'Insecta'}
temperatures = np.ones(N_CLASSES, dtype=np.float32)
for ci, label in enumerate(PRIMARY_LABELS):
    cls = CLASS_NAME_MAP.get(label, 'Aves')
    temperatures[ci] = 0.95 if cls in TEXTURE_TAXA else 1.10

print('Helpers ready')

Helpers ready


In [9]:
# MLP probes

def build_class_freq_weights(Y, cap=10.0):
    pos_count = Y.sum(axis=0).astype(np.float32) + 1.0
    freq      = pos_count / Y.shape[0]
    weights   = np.clip(1.0 / (freq ** 0.5), 1.0, cap)
    return (weights / weights.mean()).astype(np.float32)

def build_seq_features(scores_col, n_windows=N_WINDOWS):
    N = len(scores_col)
    x     = scores_col.reshape(-1, n_windows)
    prev  = np.concatenate([x[:, :1], x[:, :-1]], axis=1)
    next_ = np.concatenate([x[:, 1:], x[:, -1:]], axis=1)
    mean  = np.repeat(x.mean(axis=1), n_windows)
    max_  = np.repeat(x.max(axis=1),  n_windows)
    std   = np.repeat(x.std(axis=1),  n_windows)
    return prev.reshape(-1), next_.reshape(-1), mean, max_, std

def train_mlp_probes(emb, scores_raw, Y, min_pos=5, pca_dim=64, alpha_blend=0.25):
    scaler = StandardScaler()
    emb_s  = scaler.fit_transform(emb)
    pca    = PCA(n_components=min(pca_dim, emb_s.shape[1] - 1))
    Z      = pca.fit_transform(emb_s).astype(np.float32)
    print(f'  PCA: {emb.shape} → {Z.shape}  ({pca.explained_variance_ratio_.sum():.2%})')

    class_weights = build_class_freq_weights(Y, cap=10.0)
    probe_models  = {}
    active = np.where(Y.sum(axis=0) >= min_pos)[0]
    print(f'  Training MLP probes for {len(active)} species...')
    MAX_ROWS = 2000

    for ci in active:
        y = Y[:, ci]
        if y.sum() == 0 or y.sum() == len(y): continue
        prev, next_, mean, max_, std = build_seq_features(scores_raw[:, ci])
        X = np.hstack([Z, scores_raw[:, ci:ci+1],
                       prev[:, None], next_[:, None],
                       mean[:, None], max_[:, None], std[:, None]])
        n_pos  = int(y.sum()); pos_idx = np.where(y == 1)[0]
        w      = float(class_weights[ci])
        repeat = min(max(1, int(round(w * (len(y) - n_pos) / max(n_pos, 1)))), 8)
        if n_pos * repeat + len(y) > MAX_ROWS:
            repeat = max(1, (MAX_ROWS - len(y)) // max(n_pos, 1))
        X_bal = np.vstack([X, np.tile(X[pos_idx], (repeat, 1))])
        y_bal = np.concatenate([y, np.ones(n_pos * repeat, dtype=y.dtype)])
        clf = MLPClassifier(
            hidden_layer_sizes=(128, 64), activation='relu',
            max_iter=200, early_stopping=True,
            validation_fraction=0.15, n_iter_no_change=15,
            random_state=42, learning_rate_init=5e-4, alpha=0.005,
        )
        clf.fit(X_bal, y_bal)
        probe_models[ci] = clf

    print(f'  Trained {len(probe_models)} probes')
    return probe_models, scaler, pca, alpha_blend

def apply_mlp_probes(emb_test, scores_test, probe_models, scaler, pca, alpha_blend=0.25):
    Z_test = pca.transform(scaler.transform(emb_test)).astype(np.float32)
    result = scores_test.copy()
    for ci, clf in probe_models.items():
        prev, next_, mean, max_, std = build_seq_features(scores_test[:, ci])
        X_test = np.hstack([Z_test, scores_test[:, ci:ci+1],
                            prev[:, None], next_[:, None],
                            mean[:, None], max_[:, None], std[:, None]])
        prob  = clf.predict_proba(X_test)[:, 1].astype(np.float32)
        logit = np.log(prob + 1e-7) - np.log(1 - prob + 1e-7)
        result[:, ci] = (1 - alpha_blend) * scores_test[:, ci] + alpha_blend * logit
    return result

print('MLP probes ready')

MLP probes ready


In [10]:
# LightProtoSSM

class SelectiveSSM(nn.Module):
    def __init__(self, d_model, d_state=16, d_conv=4):
        super().__init__()
        self.d_model = d_model; self.d_state = d_state
        self.in_proj  = nn.Linear(d_model, 2 * d_model, bias=False)
        self.conv1d   = nn.Conv1d(d_model, d_model, d_conv, padding=d_conv-1, groups=d_model)
        self.dt_proj  = nn.Linear(d_model, d_model, bias=True)
        A = torch.arange(1, d_state+1, dtype=torch.float32).unsqueeze(0).expand(d_model, -1)
        self.A_log    = nn.Parameter(torch.log(A))
        self.D        = nn.Parameter(torch.ones(d_model))
        self.B_proj   = nn.Linear(d_model, d_state, bias=False)
        self.C_proj   = nn.Linear(d_model, d_state, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x):
        B_sz, T, D = x.shape
        xz = self.in_proj(x)
        x_ssm, z = xz.chunk(2, dim=-1)
        x_conv = self.conv1d(x_ssm.transpose(1,2))[:, :, :T].transpose(1,2)
        x_conv = F.silu(x_conv)
        dt = F.softplus(self.dt_proj(x_conv))
        A  = -torch.exp(self.A_log)
        B  = self.B_proj(x_conv); C = self.C_proj(x_conv)
        h  = torch.zeros(B_sz, D, self.d_state, dtype=x.dtype, device=x.device)
        ys = []
        for t in range(T):
            dA = torch.exp(A[None] * dt[:, t, :, None])
            dB = dt[:, t, :, None] * B[:, t, None, :]
            h  = h * dA + x[:, t, :, None] * dB
            ys.append((h * C[:, t, None, :]).sum(-1))
        y = torch.stack(ys, dim=1)
        return y + x * self.D[None, None, :]


class LightProtoSSM(nn.Module):
    def __init__(self, d_input=1536, d_model=128, d_state=16,
                 n_classes=234, n_windows=N_WINDOWS, dropout=0.15,
                 n_sites=20, meta_dim=16,
                 use_cross_attn=True, cross_attn_heads=2):
        super().__init__()
        self.n_classes = n_classes; self.n_windows = n_windows
        self.use_cross_attn = use_cross_attn
        self.input_proj = nn.Sequential(
            nn.Linear(d_input, d_model), nn.LayerNorm(d_model), nn.GELU(), nn.Dropout(dropout))
        self.pos_enc  = nn.Parameter(torch.randn(1, n_windows, d_model) * 0.02)
        self.site_emb = nn.Embedding(n_sites, meta_dim)
        self.hour_emb = nn.Embedding(24, meta_dim)
        self.meta_proj = nn.Linear(2 * meta_dim, d_model)
        self.ssm_fwd   = nn.ModuleList([SelectiveSSM(d_model, d_state) for _ in range(2)])
        self.ssm_bwd   = nn.ModuleList([SelectiveSSM(d_model, d_state) for _ in range(2)])
        self.ssm_merge = nn.ModuleList([nn.Linear(2 * d_model, d_model) for _ in range(2)])
        self.ssm_norm  = nn.ModuleList([nn.LayerNorm(d_model) for _ in range(2)])
        self.drop = nn.Dropout(dropout)
        if use_cross_attn:
            self.cross_attn = nn.ModuleList([
                nn.MultiheadAttention(d_model, num_heads=cross_attn_heads, dropout=dropout, batch_first=True)
                for _ in range(2)])
            self.cross_norm = nn.ModuleList([nn.LayerNorm(d_model) for _ in range(2)])
        self.prototypes  = nn.Parameter(torch.randn(n_classes, d_model) * 0.02)
        self.proto_temp  = nn.Parameter(torch.tensor(5.0))
        self.class_bias  = nn.Parameter(torch.zeros(n_classes))
        self.fusion_alpha = nn.Parameter(torch.zeros(n_classes))

    def init_prototypes(self, emb_tensor, labels_tensor):
        with torch.no_grad():
            h = self.input_proj(emb_tensor)
            for c in range(self.n_classes):
                mask = labels_tensor[:, c] > 0.5
                if mask.sum() > 0:
                    self.prototypes.data[c] = F.normalize(h[mask].mean(0), dim=0)

    def forward(self, emb, perch_logits=None, site_ids=None, hours=None):
        B, T, _ = emb.shape
        h = self.input_proj(emb) + self.pos_enc[:, :T, :]
        if site_ids is not None and hours is not None:
            site_ids = site_ids.clamp(0, self.site_emb.num_embeddings - 1)
            hours    = hours.clamp(0, 23)
            meta = self.meta_proj(torch.cat([self.site_emb(site_ids), self.hour_emb(hours)], dim=-1))
            h = h + meta[:, None, :]
        for i, (fwd, bwd, merge, norm) in enumerate(zip(self.ssm_fwd, self.ssm_bwd, self.ssm_merge, self.ssm_norm)):
            res = h
            h_f = fwd(h); h_b = bwd(h.flip(1)).flip(1)
            h   = self.drop(merge(torch.cat([h_f, h_b], dim=-1)))
            h   = norm(h + res)
            if self.use_cross_attn:
                attn_out, _ = self.cross_attn[i](h, h, h)
                h = self.cross_norm[i](h + attn_out)
        h_n = F.normalize(h, dim=-1)
        p_n = F.normalize(self.prototypes, dim=-1)
        sim = torch.matmul(h_n, p_n.T) * F.softplus(self.proto_temp) + self.class_bias[None, None, :]
        if perch_logits is not None:
            alpha = torch.sigmoid(self.fusion_alpha)[None, None, :]
            return alpha * sim + (1.0 - alpha) * perch_logits
        return sim

    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


def train_proto_ssm(emb_full, scores_full, Y_full, meta_full,
                    n_epochs=15, patience=5, lr=5e-4, n_sites=20, verbose=False):
    n_files = len(emb_full) // N_WINDOWS
    emb_f   = emb_full.reshape(n_files, N_WINDOWS, -1)
    log_f   = scores_full.reshape(n_files, N_WINDOWS, -1)
    lab_f   = Y_full.reshape(n_files, N_WINDOWS, -1).astype(np.float32)

    fnames = meta_full.groupby('filename', sort=False).size().index.tolist()
    sites_u = sorted(meta_full['site'].astype(str).unique())
    site2i  = {s: i+1 for i, s in enumerate(sites_u)}

    site_ids = np.array([
        min(site2i.get(str(meta_full.loc[meta_full['filename']==fn,'site'].iloc[0]), 0), n_sites-1)
        for fn in fnames], dtype=np.int64)
    hour_ids = np.array([
        int(meta_full.loc[meta_full['filename']==fn,'hour_utc'].iloc[0]) % 24
        for fn in fnames], dtype=np.int64)

    model = LightProtoSSM(n_classes=N_CLASSES, n_windows=N_WINDOWS, n_sites=n_sites)
    model.init_prototypes(
        torch.tensor(emb_full, dtype=torch.float32),
        torch.tensor(Y_full, dtype=torch.float32))

    emb_t  = torch.tensor(emb_f,  dtype=torch.float32)
    log_t  = torch.tensor(log_f,  dtype=torch.float32)
    lab_t  = torch.tensor(lab_f,  dtype=torch.float32)
    site_t = torch.tensor(site_ids, dtype=torch.long)
    hour_t = torch.tensor(hour_ids, dtype=torch.long)

    pos_cnt    = lab_t.sum(dim=(0,1))
    pos_weight = ((lab_t.shape[0]*lab_t.shape[1] - pos_cnt) / (pos_cnt + 1)).clamp(max=25.0)

    opt  = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)
    best_loss = float('inf'); best_state = None; wait = 0

    for ep in range(n_epochs):
        model.train()
        out  = model(emb_t, log_t, site_ids=site_t, hours=hour_t)
        loss = (F.binary_cross_entropy_with_logits(out, lab_t,
                    pos_weight=pos_weight[None, None, :])
                + 0.15 * F.mse_loss(out, log_t))
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        if loss.item() < best_loss:
            best_loss  = loss.item()
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
        if wait >= patience: break

    if best_state: model.load_state_dict(best_state)
    model.eval()
    return model, site2i

print('LightProtoSSM ready')

LightProtoSSM ready


In [11]:
# Full pipeline OOF
print('Building prior tables from sc (all labeled data)...')
prior_tables = build_prior_tables(sc, Y_SC)

file_meta = meta_tr.drop_duplicates('filename').reset_index(drop=True)
gkf = GroupKFold(n_splits=N_OOF_SPLITS)
oof_probs = np.zeros((len(sc_tr), N_CLASSES), dtype=np.float32)

fold_aucs = []

for fold, (tr_f, va_f) in enumerate(
    gkf.split(file_meta, groups=file_meta['filename']), 1
):
    t_fold = time.time()
    tr_fnames = set(file_meta.iloc[tr_f]['filename'])
    va_fnames = set(file_meta.iloc[va_f]['filename'])

    tr_mask = meta_tr['filename'].isin(tr_fnames).values
    va_mask = meta_tr['filename'].isin(va_fnames).values

    emb_tr_f  = emb_tr[tr_mask]; sc_tr_f = sc_tr[tr_mask]
    Y_tr_f    = Y_FULL_aligned[tr_mask]
    meta_tr_f = meta_tr[tr_mask].reset_index(drop=True)

    emb_va_f  = emb_tr[va_mask]; sc_va_f = sc_tr[va_mask]
    meta_va_f = meta_tr[va_mask].reset_index(drop=True)

    print(f'\nFold {fold}/{N_OOF_SPLITS} — train files: {len(tr_fnames)}, val files: {len(va_fnames)}')

    # --- ProtoSSM ---
    proto_model, site2i = train_proto_ssm(
        emb_tr_f, sc_tr_f, Y_tr_f, meta_tr_f,
        n_epochs=15, patience=5, lr=5e-4)

    n_va = len(emb_va_f) // N_WINDOWS
    va_fn_list = meta_va_f.drop_duplicates('filename')['filename'].tolist()
    va_site_ids = np.array([
        min(site2i.get(meta_va_f.loc[meta_va_f['filename']==fn,'site'].iloc[0], 0), 19)
        for fn in va_fn_list], dtype=np.int64)
    va_hour_ids = np.array([
        int(meta_va_f.loc[meta_va_f['filename']==fn,'hour_utc'].iloc[0]) % 24
        for fn in va_fn_list], dtype=np.int64)

    proto_model.eval()
    with torch.no_grad():
        proto_va = proto_model(
            torch.tensor(emb_va_f.reshape(n_va, N_WINDOWS, -1), dtype=torch.float32),
            torch.tensor(sc_va_f.reshape(n_va, N_WINDOWS, -1), dtype=torch.float32),
            site_ids=torch.tensor(va_site_ids, dtype=torch.long),
            hours=torch.tensor(va_hour_ids, dtype=torch.long),
        ).numpy().reshape(-1, N_CLASSES)

    # --- MLP probes ---
    probe_models, emb_scaler, emb_pca, alpha_blend = train_mlp_probes(
        emb_tr_f, sc_tr_f, Y_tr_f, min_pos=5, pca_dim=64)
    sc_va_mlp = apply_mlp_probes(
        emb_va_f, sc_va_f, probe_models, emb_scaler, emb_pca, alpha_blend)

    # --- Prior on val raw scores ---
    sc_va_prior = apply_prior(
        sc_va_f,
        sites=meta_va_f['site'].to_numpy(),
        hours=meta_va_f['hour_utc'].to_numpy(),
        tables=prior_tables, lambda_prior=0.10)

    # --- Ensemble ---
    first_pass = 0.20 * proto_va + 0.55 * sc_va_mlp + 0.25 * sc_va_prior

    # --- Temperature + sigmoid ---
    first_pass = first_pass / temperatures[None, :]
    probs_va   = sigmoid(first_pass)

    # --- Smoothing ---
    probs_va = file_confidence_scale(probs_va, power=0.05)
    probs_va = adaptive_delta_smooth(probs_va, base_alpha=0.10)
    probs_va = np.clip(probs_va, 0.0, 1.0)

    oof_probs[va_mask] = probs_va

    Y_va = Y_FULL_aligned[va_mask]
    fold_auc = macro_auc(Y_va, probs_va)
    fold_aucs.append(fold_auc)
    print(f'  Fold {fold} AUC={fold_auc:.5f}  time={time.time()-t_fold:.0f}s')

print(f'\nFold AUCs: {[f"{a:.5f}" for a in fold_aucs]}')
print(f'Mean fold AUC: {np.mean(fold_aucs):.5f}')

Building prior tables from sc (all labeled data)...

Fold 1/3 — train files: 39, val files: 20


  PCA: (468, 1536) → (468, 64)  (84.62%)
  Training MLP probes for 45 species...


  Trained 45 probes
  Fold 1 AUC=0.92641  time=12s

Fold 2/3 — train files: 39, val files: 20


  PCA: (468, 1536) → (468, 64)  (83.72%)
  Training MLP probes for 53 species...


  Trained 53 probes
  Fold 2 AUC=0.92656  time=11s

Fold 3/3 — train files: 40, val files: 19


  PCA: (480, 1536) → (480, 64)  (84.51%)
  Training MLP probes for 51 species...


  Trained 51 probes
  Fold 3 AUC=0.90625  time=11s

Fold AUCs: ['0.92641', '0.92656', '0.90625']
Mean fold AUC: 0.91974


In [12]:
# Final OOF scores
oof_auc  = macro_auc(Y_FULL_aligned, oof_probs)
oof_cmap = padded_cmap(Y_FULL_aligned, oof_probs)

# Baseline: raw Perch
raw_probs = sigmoid(sc_tr / temperatures[None, :])
raw_auc   = macro_auc(Y_FULL_aligned, raw_probs)
raw_cmap  = padded_cmap(Y_FULL_aligned, raw_probs)

print('=' * 60)
print(f'Raw Perch     OOF AUC={raw_auc:.5f}  cmAP={raw_cmap:.5f}')
print(f'Full pipeline OOF AUC={oof_auc:.5f}  cmAP={oof_cmap:.5f}')
print(f'Delta AUC: {oof_auc - raw_auc:+.5f}  Delta cmAP: {oof_cmap - raw_cmap:+.5f}')
print(f'Total wall time: {(time.time() - _WALL_START)/60:.1f} min')

Raw Perch     OOF AUC=0.73902  cmAP=0.86916
Full pipeline OOF AUC=0.92248  cmAP=0.91664
Delta AUC: +0.18347  Delta cmAP: +0.04748
Total wall time: 0.8 min
